## Building GeoJSON files from our annotation volumes

In [1]:
import os
import json
import plotly.graph_objects as go

from plotlybrain.build_geoJSON import (
    load_annotation_volume,
    load_structure_graph,
    build_geojson,
    scale_cartesian_to_lonlat,
)

/home/ateruels@neuro.uni-bremen.de/PlotlyBrain/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
volume, header = load_annotation_volume(resolution_um=100)
structure_df = load_structure_graph()

print("Volume shape:", volume.shape)
print("Number of ontology regions:", len(structure_df))


Volume shape: (132, 80, 114)
Number of ontology regions: 1327


In [3]:
# 2. Build one real slice
geojson = build_geojson(
    volume=volume,
    structure_df=structure_df,
    orientation="coronal",
    resolution_um=100,
    coords_mm=[-2.0],
    min_area_px=5,
    simplify_px=0.8,
    polygon_mode="contour",
    smooth_sigma=1.0,
)

Building GeoJSON slices: 100%|██████████| 1/1 [00:00<00:00, 30.61it/s]


In [4]:
geojson_lonlat = scale_cartesian_to_lonlat(geojson)

In [5]:
fig = go.Figure()

for feature in geojson_lonlat["features"]:
    region_name = feature["properties"]["Region name"]
    region_id = feature["properties"]["Region ID"]

    # optional: skip background/root-like regions
    if region_id in [0, 997]:
        continue

    for polygon in feature["geometry"]["coordinates"]:
        for ring in polygon:
            lon = [p[0] for p in ring]
            lat = [p[1] for p in ring]

            fig.add_trace(
                go.Scattergeo(
                    lon=lon,
                    lat=lat,
                    mode="lines",
                    line=dict(
                        color="black",
                        width=0.5,
                    ),
                    hovertemplate=(
                        f"<b>{region_name}</b><br>"
                        f"Region ID: {region_id}"
                        "<extra></extra>"
                    ),
                    showlegend=False,
                )
            )

fig.update_geos(
    visible=False,
    fitbounds="locations",
)

fig.update_layout(
    title="Allen CCF coronal slice, AP = -2.0 mm",
    width=700,
    height=700,
    margin=dict(l=0, r=0, t=40, b=0),
    paper_bgcolor="white",
    plot_bgcolor="white",
)

fig.show()